### Uso di PyDHM 

# pyDHM — Guida completa dalla pubblicazione scientifica
> Castañeda R, Trujillo C, Doblas A (2022) *pyDHM: A Python library for applications in digital holographic microscopy.* PLoS ONE 17(10): e0275818. https://doi.org/10.1371/journal.pone.0275818

---

## 1. Cos'è pyDHM

pyDHM è una libreria Python open-source per applicazioni di **Microscopia Olografica Digitale (DHM)**. Fornisce algoritmi di ricostruzione numerica per ottenere immagini di ampiezza e fase da un'ampia varietà di configurazioni ottiche DHM, tra cui:

- Sistemi **in-line** (on-axis)
- Sistemi **slightly off-axis**
- Sistemi **off-axis** in regime **telcentrico** e **non-telcentrico**
- Sistemi con ologrammi **a fuoco** e **fuori fuoco**

> **Installazione:** `pip install pyDHM` — richiede Python 3.7+, NumPy, OpenCV (cv2) e SciPy.

---

## 2. Background fisico: come funziona un DHM

### 2.1 Setup ottico (interferometro di Mach-Zehnder)

Un sistema DHM in trasmissione si basa su un **interferometro di Mach-Zehnder**:

1. Un laser di lunghezza d'onda λ viene collimato da una lente convergente (CL).
2. Un beam splitter (BS1) divide il fascio in **fascio oggetto (O)** e **fascio di riferimento (R)**.
3. Il fascio oggetto attraversa il campione, un obiettivo microscopico (MO) e una lente tubo (TL).
4. I due fasci si ricombinano sul sensore, generando l'**ologramma h(x,y;z)**.

### 2.2 Le tre configurazioni DHM (dal paper, Fig. 1)

La classificazione si basa sull'**angolo di interferenza** tra fascio oggetto e fascio di riferimento, visibile dallo **spettro di Fourier dell'ologramma**:

| Configurazione | Angolo θ | Spettro Fourier | Algoritmo da usare |
|---|---|---|---|
| **In-line (on-axis)** | θ = 0 | Ordini DC e ±1 completamente sovrapposti | Phase-shifting (PS3, PS4, PS5) |
| **Slightly off-axis** | θ piccolo | Ordini parzialmente sovrapposti | Phase-shifting (SOSR, BPS3, BPS2) |
| **Off-axis** | θ grande | Ordini completamente separati | Filtro spaziale + compensazione (FRS, ERS, CFS, CNT) |

### 2.3 L'equazione dell'ologramma

L'ologramma registrato è:

```
h(x,y;z) = |u(x,y;z)|² + |r(x,y)|² + u(x,y;z)·r*(x,y) + u*(x,y;z)·r(x,y)
```

dove:
- I primi due termini = irradianze dei fasci (termine DC)
- Il terzo termine = **immagine reale** del campione (+1 ordine)
- Il quarto termine = **immagine gemella** (–1 ordine)

La trasformata di Fourier dell'ologramma **H(u,v;z)** separa questi contributi in base all'angolo θ. In sistemi off-axis i tre ordini (DC, +1, –1) sono **completamente separati** nello spazio delle frequenze, permettendo la ricostruzione da un **singolo ologramma**.

### 2.4 Regime telcentrico vs. non-telcentrico

Il campo complesso prodotto dal microscopio al piano immagine (IP) contiene un fattore di fase quadratico:

```
exp(i·k/(2C) · (x² + y²))    con C = fTL² / (fTL − d)
```

- **Regime telcentrico (d = fTL):** il fattore quadratico scompare → ricostruzione semplificata (FRS, ERS, CFS).
- **Regime non-telcentrico (d ≠ fTL):** il fattore quadratico distorce la fase → serve la funzione CNT per compensarlo.

In regime non-telcentrico, gli ordini ±1 nello spettro di Fourier sono **rettangolari** (più larghi all'aumentare della curvatura), mentre in regime telcentrico sono **circolari** con diametro proporzionale all'apertura numerica NA/(λM).

---

## 3. Struttura della libreria: 4 pacchetti

---

### Pacchetto 1 — `utilities`

```python
from pyDHM import utilities
```

Funzioni di base per leggere, visualizzare immagini e operare nel dominio di Fourier.

| Funzione | Firma | Descrizione |
|---|---|---|
| `imageRead` | `imageRead(namefile)` | Legge un'immagine (ologramma) da file |
| `imageShow` | `imageShow(inp, name)` | Visualizza un'immagine con etichetta |
| `amplitude` | `amplitude(output, log)` | Calcola l'ampiezza del campo complesso. `log=True` applica scala logaritmica |
| `intensity` | `intensity(output, log)` | Calcola l'intensità (ampiezza²) del campo complesso |
| `phase` | `phase(output)` | Estrae la mappa di fase **wrapped** ∈ [−π, +π] |
| `FT` | `FT(input)` | Trasformata di Fourier 2D |
| `IFT` | `IFT(input)` | Trasformata di Fourier inversa 2D |
| `sfc` | `sfc(field, radius, centX, centY, display)` | Filtro spaziale con **maschera circolare** |
| `sfr` | `sfr(field, x1, x2, y1, y2, display)` | Filtro spaziale con **maschera rettangolare** |
| `sfmr` | `sfmr(field, display)` | Filtro rettangolare **manuale** via GUI OpenCV |
| `HM2F` | `HM2F(inp, kernel)` | Filtro ibrido mediana-media per ridurre lo **speckle noise** |

> **Nota:** `sfmr` richiede OpenCV installato. Usare per selezionare interattivamente l'ordine +1 nello spettro di Fourier.

---

### Pacchetto 2 — `phaseShifting`

```python
from pyDHM import phaseShifting
```

Ricostruzione del campo complesso per sistemi **in-line** e **slightly off-axis** tramite tecniche di **phase-shifting (PS)**.

**Principio:** si acquisiscono più ologrammi sfasando progressivamente il fascio di riferimento. La combinazione algebrica degli ologrammi elimina i termini indesiderati e isola la distribuzione di ampiezza complessa dell'oggetto.

#### Funzioni per sistemi in-line (on-axis)

| Funzione | Firma | Sfasamento | Note |
|---|---|---|---|
| `PS5` | `PS5(inp0, inp1, inp2, inp3, inp4)` | π/2 tra ologrammi | Più robusto al rumore |
| `PS4` | `PS4(inp0, inp1, inp2, inp3)` | π/2 tra ologrammi | Buon compromesso |
| `PS3` | `PS3(inp0, inp1, inp2)` | 2π/3 tra ologrammi | Meno ologrammi, più sensibile al rumore |

Le formule implementate (dal paper, Eq. 6–8):

```
PS5: φ(x,y) = atan( 2[h(3π/2) − h(π/2)] / [2h(π) − h(0) − h(2π)] )
PS4: φ(x,y) = atan( [h(3π/2) − h(π/2)] / [h(π) − h(0)] )
PS3: φ(x,y) = atan( √3 · [h(π/3) − h(5π/3)] / [h(5π/3) + h(π/3) − 2h(π)] )
```

#### Funzioni per sistemi slightly off-axis

| Funzione | Firma | Descrizione |
|---|---|---|
| `SOSR` | `SOSR(inp0, inp1, inp2, inp3, upper, wavelength, dx, dy, s=1, steps=4)` | Metodo in quadratura (De Nicola et al., 2002). Calcola automaticamente il miglior fronte d'onda digitale di riferimento tramite ROI search |
| `BPS3` | `BPS3(inp0, inp1, inp2, wavelength, dx, dy)` | Phase-shifting **cieco** a 3 frame con sfasamenti arbitrari e incogniti |
| `BPS2` | `BPS2(inp0, inp1, wavelength, dx, dy)` | Phase-shifting **cieco** a 2 frame. Richiede che gli ordini +1 e −1 non si sovrappongano nello spettro |

**Parametri comuni:** `wavelength` (µm), `dx`/`dy` = dimensione pixel sensore (µm), `upper` = booleano per la posizione dell'ordine +1 nello spettro, `s` e `steps` = parametri per la ricerca ROI.

> **Limitazione BPS3/BPS2:** validi solo per sistemi DHM in **regime telcentrico**, poiché il centro degli ordini ±1 nella trasformata di Fourier deve corrispondere a un valore massimo.

#### Esempio di codice (PS5, PS4, PS3)

```python
from pyDHM import utilities
from pyDHM import phaseShifting

# Caricamento ologrammi
inp0 = utilities.imageRead('holo1.jpg')
inp1 = utilities.imageRead('holo2.jpg')
inp2 = utilities.imageRead('holo3.jpg')
inp3 = utilities.imageRead('holo4.jpg')
inp4 = utilities.imageRead('holo5.jpg')

# Phase-shifting
output = phaseShifting.PS5(inp0, inp1, inp2, inp3, inp4)
# oppure: phaseShifting.PS4(inp0, inp1, inp2, inp3)
# oppure: phaseShifting.PS3(inp0, inp1, inp2)

# Visualizzazione fase
phase = utilities.phase(output)
utilities.imageShow(phase, 'Phase reconstruction')
```

#### Esempio slightly off-axis (SOSR, BPS3, BPS2)

```python
from pyDHM import utilities, phaseShifting

inp0 = utilities.imageRead('holo1.jpg')
inp1 = utilities.imageRead('holo2.jpg')
inp2 = utilities.imageRead('holo3.jpg')
inp3 = utilities.imageRead('holo4.jpg')

# SOSR: λ=633nm, dx=dy=6.9µm
output = phaseShifting.SOSR(inp0, inp1, inp2, inp3, 633e-9, 6.9e-6, 6.9e-6, 1, 4)

# BPS3: λ=0.532µm, dx=dy=2.9µm
output = phaseShifting.BPS3(inp0, inp1, inp2, 0.532, 2.9, 2.9)

# BPS2
output = phaseShifting.BPS2(inp0, inp1, 0.532, 2.9, 2.9)

phase = utilities.phase(output)
utilities.imageShow(phase, 'Phase reconstruction')
```

---

### Pacchetto 3 — `phaseCompensation`

```python
from pyDHM import phaseCompensation
```

Ricostruzione di fase **completamente compensata** per sistemi **off-axis** (da un singolo ologramma). L'obiettivo è eliminare la rampa di fase lineare introdotta dall'angolo di tilt tra fascio oggetto e fascio di riferimento, ottenendo immagini senza le fastidiose **frange a dente di sega (sawtooth fringes)**.

**Pipeline di ricostruzione off-axis (2 passi):**
1. **Filtraggio spaziale** dell'ordine +1 nello spettro di Fourier dell'ologramma.
2. **Compensazione dell'angolo di interferenza** moltiplicando per una replica digitale del fronte d'onda di riferimento `r_D(x)`, con parametri ottimali cercati automaticamente.

#### Funzioni per regime telcentrico

| Funzione | Firma | Metodo di ricerca |
|---|---|---|
| `FRS` | `FRS(inp, upper, wavelength, dx, dy, s=2, step=10)` | **Full ROI search**: loop annidati su tutta la regione di interesse intorno all'ordine +1. Metriche di discontinuità di fase per trovare il migliore compensato. |
| `ERS` | `ERS(inp, upper, wavelength, dx, dy, s=5, step=0.2)` | **Efficient ROI search**: strategia euristica, percorre solo i punti più promettenti nella ROI. Computazionalmente più leggero di FRS. |
| `CFS` | `CFS(inp, wavelength, dx, dy)` | **Cost-function search**: ottimizzazione non lineare (scipy minimize) che minimizza l'inverso della metrica di discontinuità. Nessuna ROI manuale. |

#### Funzione per regime non-telcentrico

| Funzione | Firma | Descrizione |
|---|---|---|
| `CNT` | `CNT(inp, wavelength, dx, dy, x1, x2, y1, y2, spatialFilter)` | Compensa anche il **fattore di fase quadratico** del regime non-telcentrico. `spatialFilter='sfmr'` per filtro manuale, `'sfr'` per filtro rettangolare con coordinate `(x1,y1,x2,y2)`. |

> **Parametri `s` e `step`:** determinano la dimensione della ROI e la densità dei punti di ricerca. La ROI ha dimensione `(1+2s)×(1+2s)` pixel; i punti di ricerca in ogni dimensione sono `step`. Aumentare questi valori migliora la precisione ma aumenta il costo computazionale.

#### Esempio di codice (regimi telcentrico e non-telcentrico)

```python
from pyDHM import utilities, phaseCompensation

inp = utilities.imageRead('hologram.jpg')

# --- Regime TELCENTRICO ---
# FRS: λ=633nm, dx=dy=6.9µm
output = phaseCompensation.FRS(inp, True, 0.633, 6.9, 6.9, 2, 10)

# ERS
output = phaseCompensation.ERS(inp, True, 0.633, 6.9, 6.9, 5, 0.2)

# CFS: λ=532nm, dx=dy=2.6µm
output = phaseCompensation.CFS(inp, 0.532, 2.6, 2.6)

# --- Regime NON-TELCENTRICO ---
# Passo 1: calcola FT per trovare coordinate del filtro
ft_holo = utilities.FT(inp)
utilities.imageShow(utilities.intensity(ft_holo, True), 'FT hologram')

# Passo 2: CNT con filtro rettangolare
output = phaseCompensation.CNT(inp, 0.633, 6.9, 6.9, 287, 180, 267, 200, spatialFilter='sfr')
# oppure con filtro manuale GUI:
output = phaseCompensation.CNT(inp, 0.633, 6.9, 6.9, spatialFilter='sfmr')

phase = utilities.phase(output)
utilities.imageShow(phase, 'Phase reconstruction')
```

> **Note sulla funzione CNT:** dopo il filtraggio, la funzione mostra un'immagine binarizzata della fase e chiede all'utente di inserire le coordinate del centro del fattore di fase quadratico (`X_cent`, `Y_cent`). Il raggio di curvatura C viene stimato automaticamente dalla dimensione della maschera rettangolare.

---

### Pacchetto 4 — `numericalPropagation`

```python
from pyDHM import numericalPropagation
```

Propagazione numerica di distribuzioni di ampiezza complessa. Utilizzata per **mettere a fuoco digitalmente** ologrammi registrati **fuori dal piano immagine** del microscopio.

**Principio fisico:** se l'ologramma è registrato a distanza z dal piano immagine (IP), il campo complesso deve essere propagato numericamente per ottenere l'immagine a fuoco. La propagazione risolve l'equazione di diffrazione di Fresnel-Kirchhoff.

| Propagatore | Firma | Equazione implementata | Uso ottimale |
|---|---|---|---|
| `angularSpectrum` | `angularSpectrum(field, z, wavelength, dx, dy)` | Eq. 11: decomposizione in onde piane (principio di Huygens) | Distanze di propagazione **brevi** |
| `fresnel` | `fresnel(field, z, wavelength, dx, dy)` | Eq. 12: approssimazione parassiale della diffrazione | Distanze di propagazione **grandi** |
| `bluestein` | `bluestein(field, z, wavelength, dx, dy, dxout, dyout)` | Variante Fresnel con sostituzione di Bluestein: conversione in operazione di convoluzione | Qualsiasi distanza, con **magnification variabile** |

> **Vantaggio di `bluestein`:** i parametri `dxout` e `dyout` (dimensioni pixel al piano di uscita) possono essere **diversi** da `dx` e `dy`, permettendo di controllare la magnification dell'immagine ricostruita. Es: `dxout = 14.8µm` vs `dx = 7.4µm` → magnification 2×.

#### Esempio di codice (angular spectrum)

```python
from pyDHM import utilities, numericalPropagation

# Caricamento hologramma fuori fuoco
input = utilities.imageRead('hologram.tif')
utilities.imageShow(input, 'Out-of-focus Hologram')

# Calcolo e visualizzazione spettro
ft_holo = utilities.FT(input)
ft_intensity = utilities.intensity(ft_holo, True)
utilities.imageShow(ft_intensity, 'FT hologram')

# Filtraggio spaziale (maschera circolare)
filter = utilities.sfc(input, 160, 303, 276, True)

# Propagazione: z=3.3cm, λ=633nm, dx=dy=6.9µm
output = numericalPropagation.angularSpectrum(filter, 3.3e-2, 633e-9, 6.9e-6, 6.9e-6)

# Visualizzazione intensità
intensity = utilities.intensity(output, False)
utilities.imageShow(intensity, 'Output field')
```

#### Esempio Fresnel e Fresnel-Bluestein

```python
from pyDHM import utilities, numericalPropagation

input = utilities.imageRead('hologram.bmp')

# Filtraggio rettangolare
filter = utilities.sfr(input, 280, 500, 150, 340, True)

# Fresnel: z=-45cm, λ=633nm, dx=dy=5µm
output = numericalPropagation.fresnel(filter, -450e-3, 633e-9, 5e-6, 5e-6)

# Bluestein: z=3cm, λ=633nm, dx=dy=7.4µm, dxout=dyout=14.8µm (magnif. 2×)
output = numericalPropagation.bluestein(filter, 0.03, 633e-9, 7.4e-6, 7.4e-6, 14.8e-5, 14.8e-5)

intensity = utilities.intensity(output, False)
utilities.imageShow(intensity, 'Output field')
```

---

## 4. Come scegliere l'algoritmo corretto

```
Ologramma DHM
│
├─ Spettro Fourier: gli ordini DC e ±1 si SOVRAPPONGONO?
│   │
│   ├─ Sì, completamente → Sistema IN-LINE
│   │   └─ Usa: PS3, PS4, PS5
│   │
│   ├─ Parzialmente → Sistema SLIGHTLY OFF-AXIS
│   │   ├─ Sfasamenti noti → SOSR
│   │   └─ Sfasamenti ignoti → BPS3 o BPS2
│   │
│   └─ No, completamente separati → Sistema OFF-AXIS
│       │
│       ├─ Regime TELCENTRICO (d = fTL)?
│       │   ├─ Sì → FRS, ERS, o CFS
│       │   └─ No (non-telcentrico) → CNT
│       │
│       └─ Ologramma a fuoco o fuori fuoco?
│           ├─ A fuoco → Nessuna propagazione necessaria
│           └─ Fuori fuoco → angularSpectrum, fresnel, o bluestein
```

---

## 5. Fase Wrapped e Unwrapped

### 5.1 Fase Wrapped (output di `utilities.phase()`)

La funzione `phase()` restituisce la fase estratta via `atan2(Im[u], Re[u])`, il cui range è **[−π, +π] radianti**. Questa fase è detta **wrapped** perché, quando il valore reale della fase supera ±π, la funzione `atan2` "resetta" intorno di 2π, generando **salti discontinui artificiali** nell'immagine.

Visivamente, la fase wrapped appare come una mappa con **frange a dente di sega** (sawtooth pattern), dove ogni banda chiara-scura corrisponde a un ciclo di 2π.

### 5.2 Fase Unwrapped

L'**unwrapping** è il processo che ricostruisce la fase continua reale, individuando e correggendo i salti di 2π tra pixel adiacenti. Il risultato è una mappa di fase **continua** che può essere convertita in altezza ottica o spessore:

```
h(x,y) [nm] = φ_unwrapped(x,y) / (2π) × λ [nm]
```

### 5.3 Relazione con la compensazione di fase

Nel workflow off-axis di pyDHM, le funzioni FRS/ERS/CFS/CNT eliminano la **rampa di fase lineare** dovuta all'inclinazione del fascio di riferimento (che si manifesta come frange lineari). Dopo la compensazione, l'immagine wrapped mostra solo la fase del campione. L'unwrapping successivo rimuove i salti residui:

```python
from skimage.restoration import unwrap_phase
import numpy as np

# phase_wrapped = utilities.phase(output) dopo compensazione
phase_unwrapped = unwrap_phase(phase_wrapped)

# Conversione in nanometri di cammino ottico (λ in nm)
optical_path_nm = phase_unwrapped / (2 * np.pi) * 532.0
```

---

## 6. Sistema di gestione degli errori

La libreria include messaggi di errore automatici per le seguenti situazioni:

| Condizione di errore | Messaggio |
|---|---|
| PS3/PS4/PS5 applicati a ologramma off-axis | `PS3/PS4/PS5 algorithm requires on-axis holograms` |
| FRS/ERS/CFS applicati a ologramma in-line | `FRS/ERS/CFS algorithm requires off-axis holograms` |
| CNT con opzione `sfmr` ma coordinate `(x1,y1,x2,y2)` inserite | `Please use the sfr option...` |
| CNT con opzione `sfr` ma senza coordinate | `Please use the sfmr option or insert the values for (x1,y1,x2,y2)...` |
| scipy non installato | `Please install scipy library and import the function minimize` |
| OpenCV non installato | `Please install OpenCV library before using the sfmr function` |

> La funzione interna `regime` controlla automaticamente la configurazione ottica del DHM dall'analisi dello spettro dell'ologramma e verifica che l'algoritmo chiamato sia compatibile.

---

## 7. Validazione sperimentale (dal paper)

| Esperimento | Funzione testata | Campione | Parametri DHM |
|---|---|---|---|
| In-line PS | PS5, PS4, PS3 | Phantom simulato | Ologramma simulato, sfasamento π/2 |
| Slightly off-axis | SOSR | Lente di Fresnel | λ=633nm, dx=dy=6.9µm, sfasamento π/2 con lente liquida |
| Slightly off-axis blind | BPS3, BPS2 | USAF test target (fase) | λ=0.532µm, dx=dy=2.9µm |
| Off-axis telcentrico | FRS, ERS | Sezione testa *Drosophila melanogaster* | λ=633nm, dx=dy=6.9µm |
| Off-axis telcentrico | CFS | Star target | λ=532nm, dx=dy=2.6µm |
| Off-axis non-telcentrico | CNT | *Drosophila melanogaster* | λ=633nm, dx=dy=6.9µm |
| Propagazione | angularSpectrum | USAF test target | λ=633nm, dx=dy=6.9µm, z da 0 a 3.3cm |
| Propagazione | fresnel | Modello di cavallo | λ=633nm, dx=dy=5µm, z=45cm |
| Propagazione | bluestein | Dado 1cm | λ=633nm, dx=dy=7.4µm, z=30cm, magnif. 2× e 2.5× |

---

## 8. Sviluppi futuri dichiarati dagli autori

- Algoritmo automatico per ricostruzione **senza conoscenza a priori** della configurazione DHM (solo ologramma, λ, pixel size).
- **GUI** (Graphical User Interface) per utenti senza background in ottica o programmazione.
- Supporto per **sequenze video** di ologrammi con ottimizzazione della selezione del filtro.
- Analisi biologica avanzata: **tracking** di microrganismi, **conteggio cellulare**.
- Riduzione dei tempi di elaborazione tramite implementazione **GPU**.

---

## 9. Citazione

```bibtex
@article{castaneda2022pyDHM,
  title     = {pyDHM: A Python library for applications in digital holographic microscopy},
  author    = {Castañeda, Raul and Trujillo, Carlos and Doblas, Ana},
  journal   = {PLoS ONE},
  volume    = {17},
  number    = {10},
  pages     = {e0275818},
  year      = {2022},
  doi       = {10.1371/journal.pone.0275818}
}
```

---

## 10. Risorse

- **GitHub:** https://github.com/catrujilla/pyDHM
- **Documentazione:** https://catrujilla.github.io/pyDHM/
- **Video installazione:** https://youtu.be/h76nZM6JpXo
- **Video angular spectrum:** https://youtu.be/CMHbF0uoWDk
- **Video BPS2:** https://youtu.be/Z9o0ODe1lUQ